In [1]:
# Imports and file paths 
from pathlib import Path
import re

import numpy as np
import pandas as pd
import geopandas as gpd

In [2]:
# Change this if your files are in another folder.
DATA_DIR = Path("C:/Users/htunlong/OneDrive - UGent/Research/Postdoc/Fundings/VMM/Data/Data_Module_5/Module 5")

MPS_CSV = DATA_DIR / "MPS meetdata 2017.csv"
MPS_GPKG = DATA_DIR / "MPS_stations.gpkg"

if not MPS_CSV.exists():
    raise FileNotFoundError(f"File not found: {MPS_CSV}")

In [3]:
# Read the raw MPS CSV file

def read_mps_csv_as_text(path):
    """
    Read the raw MPS CSV  as text.

    The file uses:
    - semicolon separator
    - Latin-1 style encoding
    - comma decimal notation

    At this stage we read values as text because Step 1 is only about
    station anchoring and coverage, not yet numeric correction.
    """
    return pd.read_csv(
        path,
        sep=";",
        encoding="latin1",
        dtype=str,
        na_values=["", " "],
        keep_default_na=True,
    )


raw_mps = read_mps_csv_as_text(MPS_CSV)


In [4]:
print("Raw MPS shape:", raw_mps.shape)

Raw MPS shape: (152415, 98)


In [5]:
# First 5 columns
print(raw_mps.columns[:5].tolist())

['Datum', 'Tijd (Europe/Amsterdam)', 'MP01/CH101_mwf - meetwaarde [cmH2O]', 'MP01/CH102_mwf - meetwaarde [°C]', 'MP01/CH103_mwf - meetwaarde []']


In [6]:
raw_mps.head()

,Datum,Tijd (Europe/Amsterdam),MP01/CH101_mwf - meetwaarde [cmH2O],MP01/CH102_mwf - meetwaarde [°C],MP01/CH103_mwf - meetwaarde [],MP01/CH104_mwf - meetwaarde [mV],MP01/CH105_mwf - meetwaarde [mS/cm],MP01/CH106_mwf - meetwaarde [mS/cm],MP01/CH107_mwf - meetwaarde [mS/cm],MP01/CH108_mwf - meetwaarde [?·cm],...,MP04/CH107_mwf - meetwaarde [mS/cm],MP04/CH108_mwf - meetwaarde [?·cm],MP04/CH109_mwf - meetwaarde [PSU],MP04/CH110_mwf - meetwaarde [mg/l],MP04/CH111_mwf - meetwaarde [st],MP04/CH112_mwf - meetwaarde [mg/l],MP04/CH113_mwf - meetwaarde [%],MP04/CH114_mwf - meetwaarde [mg/l],MP04/CH115_mwf - meetwaarde [mg/l],MP04/CH121_mwf - meetwaarde [cmH2O]
0,10/02/2017,16:30:41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10/02/2017,16:40:41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10/02/2017,16:50:40,1.020,"17,5","7,66","219,8",252,266,294,"3,97",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10/02/2017,16:50:41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10/02/2017,17:00:40,1.020,"17,5","7,65","215,9",252,266,294,"3,97",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Add timestamp 

def add_timestamp_column(df):
    """
    Create one timestamp column from the date and time columns.
    """
    date_col = "Datum"
    time_col = "Tijd (Europe/Amsterdam)"

    if date_col not in df.columns or time_col not in df.columns:
        raise ValueError("Expected date/time columns were not found in the MPS file.")

    timestamp_text = df[date_col].astype(str) + " " + df[time_col].astype(str)

    result = df.copy()
    result["timestamp"] = pd.to_datetime(
        timestamp_text,
        dayfirst=True,
        errors="coerce",
    )

    return result




In [8]:

mps = add_timestamp_column(raw_mps)
print("Rows:", len(mps))
print("Columns including timestamp:", len(mps.columns))

mps.head()


Rows: 152415
Columns including timestamp: 99


,Datum,Tijd (Europe/Amsterdam),MP01/CH101_mwf - meetwaarde [cmH2O],MP01/CH102_mwf - meetwaarde [°C],MP01/CH103_mwf - meetwaarde [],MP01/CH104_mwf - meetwaarde [mV],MP01/CH105_mwf - meetwaarde [mS/cm],MP01/CH106_mwf - meetwaarde [mS/cm],MP01/CH107_mwf - meetwaarde [mS/cm],MP01/CH108_mwf - meetwaarde [?·cm],...,MP04/CH108_mwf - meetwaarde [?·cm],MP04/CH109_mwf - meetwaarde [PSU],MP04/CH110_mwf - meetwaarde [mg/l],MP04/CH111_mwf - meetwaarde [st],MP04/CH112_mwf - meetwaarde [mg/l],MP04/CH113_mwf - meetwaarde [%],MP04/CH114_mwf - meetwaarde [mg/l],MP04/CH115_mwf - meetwaarde [mg/l],MP04/CH121_mwf - meetwaarde [cmH2O],timestamp
0,10/02/2017,16:30:41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-02-10 16:30:41
1,10/02/2017,16:40:41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-02-10 16:40:41
2,10/02/2017,16:50:40,1.020,"17,5","7,66","219,8",252,266,294,"3,97",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-02-10 16:50:40
3,10/02/2017,16:50:41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-02-10 16:50:41
4,10/02/2017,17:00:40,1.020,"17,5","7,65","215,9",252,266,294,"3,97",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-02-10 17:00:40


In [9]:
mps.describe()

,timestamp
count,152409
mean,2017-08-07 00:49:10.412994048
min,2017-02-10 16:30:41
25%,2017-05-24 07:20:43
50%,2017-08-10 16:50:38
75%,2017-10-31 05:00:42
max,2017-12-31 23:50:45


In [10]:
#  detect MPS stations
def find_station_ids(columns):
    """
    Find all MPS station IDs from column names such as:
    MP01/CH102_mwf - meetwaarde [°C]
    """
    station_ids = []

    for column in columns:
        match = re.match(r"(MP\d+)/", column)

        if match:
            station_id = match.group(1)

            if station_id not in station_ids:
                station_ids.append(station_id)

    return station_ids


def columns_for_station(columns, station_id):
    """
    Return all raw data columns belonging to one MPS station.
    """
    prefix = station_id + "/"
    return [column for column in columns if column.startswith(prefix)]

In [11]:
station_ids = find_station_ids(mps.columns)

print("Stations found:", station_ids)
print("Invalid timestamps:", int(mps["timestamp"].isna().sum()))
print("First timestamp:", mps["timestamp"].min())
print("Last timestamp:", mps["timestamp"].max())

Stations found: ['MP01', 'MP03', 'MP02', 'MP07', 'MP06', 'MP04']
Invalid timestamps: 6
First timestamp: 2017-02-10 16:30:41
Last timestamp: 2017-12-31 23:50:45


In [12]:
# Summarise station coverage

def summarize_station_coverage(df, station_ids):
    """
    Summarize availability for each MPS station.

    A row is counted as available for a station when at least one channel
    of that station has a non-missing value.

    The raw data are roughly 10-minute measurements, but timestamp seconds
    are not always identical. So coverage and gap counts are indicators,
    not final QC decisions.
    """
    records = []
    # remove na rows
    valid_time_df = df[df["timestamp"].notna()].copy()

    for station_id in station_ids:
        station_columns = columns_for_station(valid_time_df.columns, station_id)

        if not station_columns:
            raise ValueError(f"No columns found for station {station_id}.")
        
        # remove na rows
        has_any_value = valid_time_df[station_columns].notna().any(axis=1)
        station_times = valid_time_df.loc[has_any_value, "timestamp"].sort_values()

        first_time = station_times.min()
        last_time = station_times.max()

        if pd.isna(first_time) or pd.isna(last_time):
            expected_10min_rows = 0
            coverage_pct = np.nan
            gaps_over_10min = np.nan
            max_gap_hours = np.nan
        else:
            # summarise the timespan and possible gaps due to NA data
            expected_10min_rows = int((last_time - first_time).total_seconds() // 600) + 1
            coverage_pct = 100 * len(station_times) / expected_10min_rows

            gaps = station_times.diff().dropna()
            gaps_over_10min = int((gaps > pd.Timedelta(minutes=10, seconds=30)).sum())
            max_gap_hours = gaps.max().total_seconds() / 3600 if len(gaps) else 0

        records.append({
            "mps_station_id": station_id,
            "n_raw_channels": len(station_columns),
            "n_rows_with_any_value": int(len(station_times)),
            "first_observation": first_time,
            "last_observation": last_time,
            "expected_10min_rows_between_first_last": expected_10min_rows,
            "coverage_pct_within_station_period": round(coverage_pct, 2),
            "gaps_gt_10min30s_count": gaps_over_10min,
            "max_gap_hours": round(max_gap_hours, 2),
        })

    coverage_table = pd.DataFrame(records)

    # Sort stations logically by station ID.
    coverage_table = coverage_table.sort_values("mps_station_id").reset_index(drop=True)

    return coverage_table


coverage = summarize_station_coverage(mps, station_ids)
coverage


,mps_station_id,n_raw_channels,n_rows_with_any_value,first_observation,last_observation,expected_10min_rows_between_first_last,coverage_pct_within_station_period,gaps_gt_10min30s_count,max_gap_hours
0,MP01,16,42954,2017-02-10 16:50:40,2017-12-31 23:50:41,46699,91.98,543,75.67
1,MP02,16,42559,2017-02-10 16:30:41,2017-12-31 23:50:45,46701,91.13,523,129.17
2,MP03,16,41435,2017-02-18 15:40:41,2017-12-31 23:50:45,45554,90.96,569,76.17
3,MP04,16,41798,2017-02-18 15:10:43,2017-12-31 23:50:41,45556,91.75,557,38.00
4,MP06,16,13857,2017-08-24 11:00:00,2017-12-20 11:00:44,16993,81.55,144,48.00
5,MP07,16,13956,2017-08-24 11:00:00,2017-12-20 11:50:44,16998,82.10,232,121.00


In [13]:
def read_mps_gpkg_locations(path):
    """
    Read the MPS station coordinate file.

    The GPKG is in EPSG:4326.
    We also convert it to Belgian Lambert 72, EPSG:31370,
    because the ICM hydraulic coordinates are in metres.
    """
    gdf = gpd.read_file(path)

    gdf["latitude"] = pd.to_numeric(gdf["station_latitude"], errors="coerce")
    gdf["longitude"] = pd.to_numeric(gdf["station_longitude"], errors="coerce")

    lambert = gdf.to_crs(31370)

    gdf["x_lambert72"] = lambert.geometry.x
    gdf["y_lambert72"] = lambert.geometry.y

    return gdf


def filter_project_area_stations(gdf):
    """
    Keep stations in the Dommel-Warmbeek project area.

    This bounding box is only used to keep the table readable.
    """
    in_lon_range = gdf["longitude"].between(5.20, 5.60)
    in_lat_range = gdf["latitude"].between(51.10, 51.35)

    return gdf[in_lon_range & in_lat_range].copy()


mps_gpkg = read_mps_gpkg_locations(MPS_GPKG)
project_gpkg = filter_project_area_stations(mps_gpkg)

project_gpkg[
    [
        "station_name",
        "station_no",
        "station_id",
        "longitude",
        "latitude",
        "x_lambert72",
        "y_lambert72",
    ]
].sort_values("station_name")

,station_name,station_no,station_id,longitude,latitude,x_lambert72,y_lambert72
17,Achel/Beverbekerdijk/Warmbeek,IMM5024,429432,5.492997,51.281766,228439.593473,219694.591097
171,Peer/Dijkerstraat/Dommel,IMM5010,423248,5.433813,51.135078,224544.771264,203315.703695
172,Pelt/FierkensHeikant/Leemskuilderloop,IMM5047,459005,5.492354,51.227579,228486.014360,213665.963716
173,Pelt/GroteHeide/Dommel,IMM5023,428254,5.428441,51.267877,223957.817998,218083.217117
174,Pelt/KorteDijk/Warmbeek,IMM5046,459024,5.493009,51.224813,228536.426372,213358.960908
175,Pelt/Rallylaan/Eindergatloop,IMM5033,437881,5.409227,51.235524,222667.385548,214465.126854


In [19]:
MPS_LOCATION_LOOKUP = {
    "MP01": {
        "location_name": "Goudbergstraat",
        "watercourse": "Dommel",
    },
    "MP02": {
        "location_name": "Hoksentstraat",
        "watercourse": "Dommel",
    },
    "MP03": {
        "location_name": "Watermolen van Molhem",
        "watercourse": "Dommel",
    },
    "MP04": {
        "location_name": "Warmbeek downstream of canal",
        "watercourse": "Warmbeek",
    },
    "MP06": {
        "location_name": "Warmbeek upstream of Prinsenloop",
        "watercourse": "Warmbeek",
    },
    "MP07": {
        "location_name": "Eindergatloop",
        "watercourse": "Eindergatloop",
    },
}


def build_base_mps_anchor(station_ids, location_lookup):
    """
    Build one row per MPS station using known station names.
    """
    records = []

    for station_id in station_ids:
        lookup = location_lookup[station_id]

        records.append({
            "mps_station_id": station_id,
            "location_name": lookup["location_name"],
            "watercourse": lookup["watercourse"],
        })

    return pd.DataFrame(records)


In [20]:

def build_gpkg_candidates_for_mps(mps_anchor, project_gdf):
    """
    Match MPS stations to GPKG coordinate candidates by watercourse name.

    This is deliberately conservative:
    - unique watercourse match can be filled automatically;
    - multiple matches remain ambiguous.
    """
    records = []

    for _, station in mps_anchor.iterrows():
        station_id = station["mps_station_id"]
        watercourse = station["watercourse"]

        matches = project_gdf[
            project_gdf["station_name"].str.contains(watercourse, case=False, na=False)
        ].copy()

        for _, match in matches.iterrows():
            records.append({
                "mps_station_id": station_id,
                "mps_location_name": station["location_name"],
                "mps_watercourse": watercourse,
                "gpkg_station_name": match["station_name"],
                "gpkg_station_no": match["station_no"],
                "gpkg_station_id": match["station_id"],
                "longitude": float(match["longitude"]),
                "latitude": float(match["latitude"]),
                "x_lambert72": float(match["x_lambert72"]),
                "y_lambert72": float(match["y_lambert72"]),
            })

    candidates = pd.DataFrame(records)

    candidates["coordinate_candidate_count"] = (
        candidates
        .groupby("mps_station_id")["gpkg_station_no"]
        .transform("count")
    )

    candidates["coordinate_match_status"] = np.where(
        candidates["coordinate_candidate_count"] == 1,
        "unique_watercourse_match",
        "ambiguous_watercourse_match",
    )

    return candidates.sort_values(
        ["mps_station_id", "gpkg_station_name"]
    ).reset_index(drop=True)


base_anchor = build_base_mps_anchor(station_ids, MPS_LOCATION_LOOKUP)
gpkg_candidates = build_gpkg_candidates_for_mps(base_anchor, project_gpkg)

gpkg_candidates

,mps_station_id,mps_location_name,mps_watercourse,gpkg_station_name,gpkg_station_no,gpkg_station_id,longitude,latitude,x_lambert72,y_lambert72,coordinate_candidate_count,coordinate_match_status
0,MP01,Goudbergstraat,Dommel,Peer/Dijkerstraat/Dommel,IMM5010,423248,5.433813,51.135078,224544.771264,203315.703695,2,ambiguous_watercourse_match
1,MP01,Goudbergstraat,Dommel,Pelt/GroteHeide/Dommel,IMM5023,428254,5.428441,51.267877,223957.817998,218083.217117,2,ambiguous_watercourse_match
2,MP02,Hoksentstraat,Dommel,Peer/Dijkerstraat/Dommel,IMM5010,423248,5.433813,51.135078,224544.771264,203315.703695,2,ambiguous_watercourse_match
3,MP02,Hoksentstraat,Dommel,Pelt/GroteHeide/Dommel,IMM5023,428254,5.428441,51.267877,223957.817998,218083.217117,2,ambiguous_watercourse_match
4,MP03,Watermolen van Molhem,Dommel,Peer/Dijkerstraat/Dommel,IMM5010,423248,5.433813,51.135078,224544.771264,203315.703695,2,ambiguous_watercourse_match
5,MP03,Watermolen van Molhem,Dommel,Pelt/GroteHeide/Dommel,IMM5023,428254,5.428441,51.267877,223957.817998,218083.217117,2,ambiguous_watercourse_match
6,MP04,Warmbeek downstream of canal,Warmbeek,Achel/Beverbekerdijk/Warmbeek,IMM5024,429432,5.492997,51.281766,228439.593473,219694.591097,2,ambiguous_watercourse_match
7,MP04,Warmbeek downstream of canal,Warmbeek,Pelt/KorteDijk/Warmbeek,IMM5046,459024,5.493009,51.224813,228536.426372,213358.960908,2,ambiguous_watercourse_match
8,MP06,Warmbeek upstream of Prinsenloop,Warmbeek,Achel/Beverbekerdijk/Warmbeek,IMM5024,429432,5.492997,51.281766,228439.593473,219694.591097,2,ambiguous_watercourse_match
9,MP06,Warmbeek upstream of Prinsenloop,Warmbeek,Pelt/KorteDijk/Warmbeek,IMM5046,459024,5.493009,51.224813,228536.426372,213358.960908,2,ambiguous_watercourse_match


In [22]:
# Save the Step 1 output

OUTPUT_FILE = DATA_DIR / "M5_step01_MPS_spatial_anchor.xlsx"

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    gpkg_candidates.to_excel(writer, sheet_name="MPS_Spatial_Anchor", index=False)
    coverage.to_excel(writer, sheet_name="MPS_Coverage_Check", index=False)

print(f"Saved: {OUTPUT_FILE}")

Saved: C:\Users\htunlong\OneDrive - UGent\Research\Postdoc\Fundings\VMM\Data\Data_Module_5\Module 5\M5_step01_MPS_spatial_anchor.xlsx
